# 00 — Bronze Data Copy & Dual-Backend Ingestion (TPC-DS)

This notebook ingests the 8 TPC-DS benchmark tables intra-`US` (`$0.00` egress) according to `SCALE_PRESET` in `.env` into both Bronze storage backends:

1. **Path S2 (`DS_BRONZE_NATIVE`):** Clustered native BigQuery Capacitor tables inside a `storage_billing_model = 'PHYSICAL'` dataset.
2. **Path S1 (`gs://${BENCH_BUCKET}/bronze/${SCALE_PRESET}/<table>/*.parquet` + `DS_BRONZE_BIGLAKE`):** Stale-shard-free Snappy Parquet files in Cloud Storage read directly by Dataproc Spark (`01`, `02`) and mounted via BigLake Cloud Resource External Tables (`DS_BRONZE_BIGLAKE`) for `BigFrames` (`03`).

### Scale Presets (`SPEC.md` §2) & `INGEST_MODE` Toggle
- **`small_1g`**: 100% of `bigquery-public-data.tpc_ds_1g` (fast end-to-end validation).
- **`medium_100g`**: `bigquery-public-data.tpc_ds_10t` sliced on `*_date_sk BETWEEN 2452276 AND 2452293` (18 days in 2002), active `customer` keys only, full `customer_demographics`, `item`, and `date_dim`.
- **`large_500g`**: `bigquery-public-data.tpc_ds_10t` sliced on `*_date_sk BETWEEN 2452276 AND 2452365` (90 days in 2002), active `customer` keys only, full `customer_demographics`, `item`, and `date_dim`.
- **`INGEST_MODE`** (env var, default `"both"`):
  - `"both"`: Populates `DS_BRONZE_NATIVE`, exports Snappy Parquet to GCS, mounts `DS_BRONZE_BIGLAKE`, and verifies 100% native vs. BigLake row-count parity.
  - `"native_only"`: Repopulates `DS_BRONZE_NATIVE` (`CREATE OR REPLACE TABLE ... CLUSTER BY ...`) and verifies native row counts without re-exporting GCS Parquet or recreating `DS_BRONZE_BIGLAKE` (ideal for fast `DS_BRONZE_NATIVE` reloads during BigQuery-only backfills).

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from google.cloud import bigquery, storage


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "metrics_helper.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root containing scripts/metrics_helper.py"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.metrics_helper import VALID_SCALE_PRESETS, resolve_bench_config

cfg = resolve_bench_config(repo_root=REPO_ROOT)

PROJECT_ID = cfg.get("PROJECT_ID", "").strip()
if not PROJECT_ID or PROJECT_ID == "your-gcp-project-id":
    raise ValueError(
        "PROJECT_ID must be set to a real GCP project ID in .env or environment."
    )

PROJECT_NUMBER = cfg.get("PROJECT_NUMBER", "").strip()
BENCH_BUCKET = cfg.get("BENCH_BUCKET", "").strip()
if not BENCH_BUCKET:
    if not PROJECT_NUMBER:
        proc = subprocess.run(
            [
                "gcloud",
                "projects",
                "describe",
                PROJECT_ID,
                "--format=value(projectNumber)",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        lines = [
            ln.strip()
            for ln in proc.stdout.splitlines()
            if ln.strip() and not ln.startswith("Created ")
        ]
        PROJECT_NUMBER = lines[-1] if lines else ""
    if not PROJECT_NUMBER.isdigit():
        raise ValueError(
            f"Unable to resolve numeric PROJECT_NUMBER for PROJECT_ID={PROJECT_ID!r}"
        )
    BENCH_BUCKET = f"lakehouse-bench-{PROJECT_NUMBER}"

SCALE_PRESET = cfg.get("SCALE_PRESET", "small_1g").strip()
if SCALE_PRESET not in VALID_SCALE_PRESETS:
    raise ValueError(
        f"Invalid SCALE_PRESET={SCALE_PRESET!r}; expected one of {sorted(VALID_SCALE_PRESETS)}"
    )

INGEST_MODE = (cfg.get("INGEST_MODE") or os.environ.get("INGEST_MODE") or "both").strip().lower()
if INGEST_MODE not in {"both", "native_only"}:
    raise ValueError(
        f"Invalid INGEST_MODE={INGEST_MODE!r}; expected 'both' or 'native_only'"
    )

BQ_LOCATION = cfg.get("BQ_LOCATION", "US").strip()
BQ_LOCATION_LOWER = BQ_LOCATION.lower()
BIGLAKE_CONNECTION_ID = cfg.get("BIGLAKE_CONNECTION_ID", "bench-biglake-us").strip()
DS_BRONZE_NATIVE = cfg.get("DS_BRONZE_NATIVE", "bench_bronze_native").strip()
DS_BRONZE_BIGLAKE = cfg.get("DS_BRONZE_BIGLAKE", "bench_bronze_biglake").strip()

print(f"Repository Root:      {REPO_ROOT}")
print(f"Scale Preset:         {SCALE_PRESET}")
print(f"Ingest Mode:          {INGEST_MODE}")
print(f"BigQuery Location:    {BQ_LOCATION}")
print(f"Bronze Native DS:     {DS_BRONZE_NATIVE}")
print(f"Bronze BigLake DS:    {DS_BRONZE_BIGLAKE}")
print(f"GCS Bronze Prefix:    gs://{BENCH_BUCKET}/bronze/{SCALE_PRESET}/")
print(f"BigLake Connection:   {BQ_LOCATION_LOWER}.{BIGLAKE_CONNECTION_ID}")

In [ ]:
bq_client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
gcs_client = storage.Client(project=PROJECT_ID) if INGEST_MODE == "both" else None
bucket = gcs_client.bucket(BENCH_BUCKET) if gcs_client is not None else None
if bucket is not None and not bucket.exists():
    raise RuntimeError(
        f"GCS bucket gs://{BENCH_BUCKET} does not exist. Run scripts/provision_infra.sh first."
    )

billing_check_sql = f'''
SELECT option_value
FROM `{PROJECT_ID}.region-{BQ_LOCATION_LOWER}.INFORMATION_SCHEMA.SCHEMATA_OPTIONS`
WHERE schema_name = '{DS_BRONZE_NATIVE}'
  AND option_name = 'storage_billing_model'
'''
billing_rows = list(bq_client.query(billing_check_sql).result())
billing_model = billing_rows[0]["option_value"] if billing_rows else "LOGICAL (default)"
print(f"{DS_BRONZE_NATIVE} storage_billing_model: {billing_model}")

FACT_DATE_COLUMNS: dict[str, str] = {
    "store_sales": "ss_sold_date_sk",
    "store_returns": "sr_returned_date_sk",
    "web_sales": "ws_sold_date_sk",
    "web_returns": "wr_returned_date_sk",
}

CLUSTER_KEYS: dict[str, str] = {
    "store_sales": "ss_sold_date_sk",
    "store_returns": "sr_returned_date_sk",
    "web_sales": "ws_sold_date_sk",
    "web_returns": "wr_returned_date_sk",
    "customer": "c_customer_sk",
    "customer_demographics": "cd_demo_sk",
    "item": "i_item_sk",
    "date_dim": "d_date_sk",
}

# Facts must be materialized before `customer` so sliced `customer` filters
# directly against the already-sliced Bronze native fact tables.
INGESTION_ORDER: list[str] = [
    "store_sales",
    "store_returns",
    "web_sales",
    "web_returns",
    "customer",
    "customer_demographics",
    "item",
    "date_dim",
]

SCALE_PRESET_SPECS: dict[str, dict[str, object]] = {
    "small_1g": {
        "source_dataset": "bigquery-public-data.tpc_ds_1g",
        "date_min": None,
        "date_max": None,
    },
    "medium_100g": {
        "source_dataset": "bigquery-public-data.tpc_ds_10t",
        "date_min": 2452276,
        "date_max": 2452293,
    },
    "large_500g": {
        "source_dataset": "bigquery-public-data.tpc_ds_10t",
        "date_min": 2452276,
        "date_max": 2452365,
    },
}

active_spec = SCALE_PRESET_SPECS[SCALE_PRESET]
print(f"Active source spec: {active_spec}")

In [ ]:
def build_native_copy_sql(table_name: str) -> str:
    source_dataset = str(active_spec["source_dataset"])
    date_min = active_spec["date_min"]
    date_max = active_spec["date_max"]
    cluster_col = CLUSTER_KEYS[table_name]
    dest_table = f"`{PROJECT_ID}.{DS_BRONZE_NATIVE}.{table_name}`"

    if date_min is None or date_max is None:
        return (
            f"CREATE OR REPLACE TABLE {dest_table}\n"
            f"CLUSTER BY {cluster_col} AS\n"
            f"SELECT * FROM `{source_dataset}.{table_name}`"
        )

    if table_name in FACT_DATE_COLUMNS:
        date_col = FACT_DATE_COLUMNS[table_name]
        return (
            f"CREATE OR REPLACE TABLE {dest_table}\n"
            f"CLUSTER BY {cluster_col} AS\n"
            f"SELECT *\n"
            f"FROM `{source_dataset}.{table_name}`\n"
            f"WHERE {date_col} BETWEEN {date_min} AND {date_max}"
        )

    if table_name == "customer":
        return (
            f"CREATE OR REPLACE TABLE {dest_table}\n"
            f"CLUSTER BY {cluster_col} AS\n"
            f"WITH sliced_facts AS (\n"
            f"  SELECT ss_customer_sk AS customer_sk\n"
            f"  FROM `{PROJECT_ID}.{DS_BRONZE_NATIVE}.store_sales`\n"
            f"  WHERE ss_sold_date_sk BETWEEN {date_min} AND {date_max}\n"
            f"    AND ss_customer_sk IS NOT NULL\n"
            f"  UNION ALL\n"
            f"  SELECT sr_customer_sk AS customer_sk\n"
            f"  FROM `{PROJECT_ID}.{DS_BRONZE_NATIVE}.store_returns`\n"
            f"  WHERE sr_returned_date_sk BETWEEN {date_min} AND {date_max}\n"
            f"    AND sr_customer_sk IS NOT NULL\n"
            f"  UNION ALL\n"
            f"  SELECT ws_bill_customer_sk AS customer_sk\n"
            f"  FROM `{PROJECT_ID}.{DS_BRONZE_NATIVE}.web_sales`\n"
            f"  WHERE ws_sold_date_sk BETWEEN {date_min} AND {date_max}\n"
            f"    AND ws_bill_customer_sk IS NOT NULL\n"
            f"  UNION ALL\n"
            f"  SELECT wr_refunded_customer_sk AS customer_sk\n"
            f"  FROM `{PROJECT_ID}.{DS_BRONZE_NATIVE}.web_returns`\n"
            f"  WHERE wr_returned_date_sk BETWEEN {date_min} AND {date_max}\n"
            f"    AND wr_refunded_customer_sk IS NOT NULL\n"
            f")\n"
            f"SELECT c.*\n"
            f"FROM `{source_dataset}.customer` AS c\n"
            f"WHERE c.c_customer_sk IN (SELECT DISTINCT customer_sk FROM sliced_facts)"
        )

    return (
        f"CREATE OR REPLACE TABLE {dest_table}\n"
        f"CLUSTER BY {cluster_col} AS\n"
        f"SELECT * FROM `{source_dataset}.{table_name}`"
    )


def clear_gcs_prefix(prefix: str) -> int:
    assert gcs_client is not None and bucket is not None
    blobs = list(gcs_client.list_blobs(BENCH_BUCKET, prefix=prefix))
    deleted = 0
    for idx in range(0, len(blobs), 500):
        chunk = blobs[idx : idx + 500]
        if chunk:
            bucket.delete_blobs(chunk)
            deleted += len(chunk)
    return deleted


def ingest_table(table_name: str) -> dict[str, object]:
    # 1. Populate Path S2 (DS_BRONZE_NATIVE)
    t0 = time.perf_counter()
    native_sql = build_native_copy_sql(table_name)
    native_job = bq_client.query(native_sql)
    native_job.result()
    native_sec = round(time.perf_counter() - t0, 2)

    if INGEST_MODE == "native_only":
        return {
            "table_name": table_name,
            "native_copy_sec": native_sec,
            "deleted_stale_blobs": 0,
            "export_parquet_sec": 0.0,
            "biglake_mount_sec": 0.0,
            "parquet_shards": 0,
            "gcs_parquet_bytes": 0,
        }

    # 2. Clear stale GCS Parquet shards under bronze/{SCALE_PRESET}/{table_name}/
    gcs_prefix = f"bronze/{SCALE_PRESET}/{table_name}/"
    deleted_blobs = clear_gcs_prefix(gcs_prefix)

    # 3. Export Native -> Open Snappy Parquet in GCS (Path S1)
    t1 = time.perf_counter()
    export_uri = f"gs://{BENCH_BUCKET}/{gcs_prefix}*.parquet"
    export_sql = (
        f"EXPORT DATA OPTIONS(\n"
        f"  uri='{export_uri}',\n"
        f"  format='PARQUET',\n"
        f"  compression='SNAPPY',\n"
        f"  overwrite=true\n"
        f") AS\n"
        f"SELECT * FROM `{PROJECT_ID}.{DS_BRONZE_NATIVE}.{table_name}`"
    )
    export_job = bq_client.query(export_sql)
    export_job.result()
    export_sec = round(time.perf_counter() - t1, 2)

    # 4. Mount BigLake External Table (DS_BRONZE_BIGLAKE)
    t2 = time.perf_counter()
    connection_ref = f"{PROJECT_ID}.{BQ_LOCATION_LOWER}.{BIGLAKE_CONNECTION_ID}"
    biglake_sql = (
        f"CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DS_BRONZE_BIGLAKE}.{table_name}`\n"
        f"WITH CONNECTION `{connection_ref}`\n"
        f"OPTIONS(\n"
        f"  format='PARQUET',\n"
        f"  uris=['{export_uri}']\n"
        f")"
    )
    biglake_job = bq_client.query(biglake_sql)
    biglake_job.result()
    biglake_sec = round(time.perf_counter() - t2, 2)

    assert gcs_client is not None
    parquet_blobs = [
        b
        for b in gcs_client.list_blobs(BENCH_BUCKET, prefix=gcs_prefix)
        if b.name.endswith(".parquet")
    ]
    gcs_bytes = sum(int(b.size or 0) for b in parquet_blobs)

    return {
        "table_name": table_name,
        "native_copy_sec": native_sec,
        "deleted_stale_blobs": deleted_blobs,
        "export_parquet_sec": export_sec,
        "biglake_mount_sec": biglake_sec,
        "parquet_shards": len(parquet_blobs),
        "gcs_parquet_bytes": gcs_bytes,
    }

In [ ]:
ingestion_results: list[dict[str, object]] = []
for tbl in INGESTION_ORDER:
    print(f"Ingesting {tbl} ({SCALE_PRESET}, mode={INGEST_MODE})...")
    stats = ingest_table(tbl)
    ingestion_results.append(stats)
    if INGEST_MODE == "native_only":
        print(
            f"  -> native={stats['native_copy_sec']}s "
            "(INGEST_MODE=native_only; skipped GCS export & BigLake mount)"
        )
    else:
        print(
            f"  -> native={stats['native_copy_sec']}s | "
            f"export={stats['export_parquet_sec']}s | "
            f"shards={stats['parquet_shards']} | "
            f"parquet_mb={stats['gcs_parquet_bytes'] / (1024 ** 2):.2f} MiB"
        )

ingestion_df = pd.DataFrame(ingestion_results)
ingestion_df

In [ ]:
def fetch_dataset_row_counts(dataset_name: str, tables: list[str]) -> dict[str, int]:
    union_parts = [
        f"SELECT '{tbl}' AS table_name, COUNT(*) AS row_count FROM `{PROJECT_ID}.{dataset_name}.{tbl}`"
        for tbl in tables
    ]
    sql = "\nUNION ALL\n".join(union_parts)
    rows = bq_client.query(sql).result()
    return {row["table_name"]: int(row["row_count"]) for row in rows}


native_counts = fetch_dataset_row_counts(DS_BRONZE_NATIVE, INGESTION_ORDER)
biglake_counts = (
    fetch_dataset_row_counts(DS_BRONZE_BIGLAKE, INGESTION_ORDER)
    if INGEST_MODE == "both"
    else {}
)

storage_sql = f'''
SELECT
  table_name,
  total_logical_bytes,
  total_physical_bytes,
  active_physical_bytes,
  long_term_physical_bytes
FROM `{PROJECT_ID}.region-{BQ_LOCATION_LOWER}.INFORMATION_SCHEMA.TABLE_STORAGE`
WHERE table_schema = '{DS_BRONZE_NATIVE}'
'''
storage_by_table = {
    row["table_name"]: dict(row.items())
    for row in bq_client.query(storage_sql).result()
}
gcs_bytes_by_table = {
    row["table_name"]: int(row["gcs_parquet_bytes"]) for row in ingestion_results
}

summary_rows: list[dict[str, object]] = []
for tbl in INGESTION_ORDER:
    n_cnt = native_counts[tbl]
    assert n_cnt > 0, f"Expected non-zero rows in {DS_BRONZE_NATIVE}.{tbl}, got {n_cnt}"

    if INGEST_MODE == "both":
        b_cnt = biglake_counts[tbl]
        assert (
            n_cnt == b_cnt
        ), f"Row count mismatch for {tbl}: native={n_cnt} vs biglake={b_cnt}"
        parity_ok: bool | None = n_cnt == b_cnt
        gcs_mib: float | None = round(gcs_bytes_by_table.get(tbl, 0) / (1024 ** 2), 2)
    else:
        b_cnt = None
        parity_ok = None
        gcs_mib = None

    st = storage_by_table.get(tbl, {})
    logical_bytes = int(st.get("total_logical_bytes") or 0)
    physical_bytes = int(st.get("total_physical_bytes") or 0)
    comp_ratio = (
        round(logical_bytes / physical_bytes, 2) if physical_bytes > 0 else None
    )

    summary_rows.append(
        {
            "table_name": tbl,
            "native_rows": n_cnt,
            "biglake_rows": b_cnt,
            "parity_ok": parity_ok,
            "bq_logical_mib": round(logical_bytes / (1024 ** 2), 2),
            "bq_physical_mib": round(physical_bytes / (1024 ** 2), 2),
            "bq_compression_x": comp_ratio,
            "gcs_parquet_mib": gcs_mib,
        }
    )

parity_df = pd.DataFrame(summary_rows)
if INGEST_MODE == "both":
    print("PASS: 100% row-count parity verified between DS_BRONZE_NATIVE and DS_BRONZE_BIGLAKE.")
else:
    print(
        f"PASS: Verified non-zero native row counts across all {len(INGESTION_ORDER)} tables "
        f"in {DS_BRONZE_NATIVE} (INGEST_MODE=native_only; BigLake parity skipped)."
    )
parity_df